In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from difflib import SequenceMatcher

# ============================================================
# 分析特定两个文档的相似度
# ============================================================

def analyze_document_pair(docno1, docno2, result_df):
    """
    详细分析两个文档的相似度
    
    Args:
        docno1, docno2: 要比较的两个文档ID
        result_df: 包含passage_text的DataFrame
    """
    print("="*80)
    print(f"ANALYZING DOCUMENTS: {docno1} vs {docno2}")
    print("="*80)
    
    # 查找这两个文档
    doc1_rows = result_df[result_df['docno'] == str(docno1)]
    doc2_rows = result_df[result_df['docno'] == str(docno2)]
    
    if len(doc1_rows) == 0:
        print(f"⚠️  Document {docno1} not found in result")
        return
    if len(doc2_rows) == 0:
        print(f"⚠️  Document {docno2} not found in result")
        return
    
    doc1 = doc1_rows.iloc[0]
    doc2 = doc2_rows.iloc[0]
    
    text1 = doc1['passage_text']
    text2 = doc2['passage_text']
    
    # 基本信息
    print("\n" + "-"*80)
    print("DOCUMENT 1:")
    print("-"*80)
    print(f"DocNo: {doc1['docno']}")
    print(f"QID: {doc1['qid']}")
    print(f"Rank: {doc1['rank']}")
    print(f"Label: {doc1['origin_label']}")
    print(f"Text length: {len(text1)} characters")
    print(f"\nFull text:\n{text1}")
    
    print("\n" + "-"*80)
    print("DOCUMENT 2:")
    print("-"*80)
    print(f"DocNo: {doc2['docno']}")
    print(f"QID: {doc2['qid']}")
    print(f"Rank: {doc2['rank']}")
    print(f"Label: {doc2['origin_label']}")
    print(f"Text length: {len(text2)} characters")
    print(f"\nFull text:\n{text2}")
    
    # 检查是否在同一个查询中
    print("\n" + "="*80)
    print("CONTEXT CHECK:")
    print("="*80)
    
    if doc1['qid'] == doc2['qid']:
        print(f"✓ Both documents are in the SAME query: {doc1['qid']}")
        same_query = True
    else:
        print(f"✗ Documents are in DIFFERENT queries:")
        print(f"  Doc1 in query: {doc1['qid']}")
        print(f"  Doc2 in query: {doc2['qid']}")
        print(f"\n⚠️  IMPORTANT: Deduplication only works WITHIN the same query!")
        print(f"     Documents in different queries are never compared.")
        same_query = False
    
    # 相似度分析
    print("\n" + "="*80)
    print("SIMILARITY ANALYSIS:")
    print("="*80)
    
    # 1. 字符级相似度 (SequenceMatcher)
    seq_similarity = SequenceMatcher(None, text1, text2).ratio()
    print(f"\n1. Character-level similarity (SequenceMatcher): {seq_similarity:.4f} ({seq_similarity*100:.2f}%)")
    
    # 2. TF-IDF余弦相似度
    try:
        vectorizer = TfidfVectorizer(min_df=1, stop_words='english')
        tfidf_matrix = vectorizer.fit_transform([text1, text2])
        tfidf_similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
        print(f"2. TF-IDF cosine similarity: {tfidf_similarity:.4f} ({tfidf_similarity*100:.2f}%)")
    except Exception as e:
        print(f"2. TF-IDF cosine similarity: Error - {e}")
        tfidf_similarity = None
    
    # 3. 简单的词重叠
    words1 = set(text1.lower().split())
    words2 = set(text2.lower().split())
    intersection = words1 & words2
    union = words1 | words2
    jaccard = len(intersection) / len(union) if len(union) > 0 else 0
    print(f"3. Jaccard similarity (word-level): {jaccard:.4f} ({jaccard*100:.2f}%)")
    print(f"   Words in common: {len(intersection)} / {len(union)} total unique words")
    
    # 4. 前缀和后缀分析
    print("\n" + "-"*80)
    print("PREFIX/SUFFIX ANALYSIS:")
    print("-"*80)
    
    # 找出不同的部分
    min_len = min(len(text1), len(text2))
    
    # 找前缀差异
    prefix_diff_idx = 0
    for i in range(min_len):
        if text1[i] != text2[i]:
            prefix_diff_idx = i
            break
    
    # 找后缀差异
    suffix_diff_idx = min_len
    for i in range(1, min_len + 1):
        if text1[-i] != text2[-i]:
            suffix_diff_idx = min_len - i + 1
            break
    
    if prefix_diff_idx > 0:
        print(f"\nCommon prefix ({prefix_diff_idx} chars):")
        print(f'  "{text1[:prefix_diff_idx]}"')
    
    if prefix_diff_idx < min_len:
        print(f"\nFirst difference at position {prefix_diff_idx}:")
        start = max(0, prefix_diff_idx - 20)
        end = min(len(text1), prefix_diff_idx + 80)
        print(f'  Doc1: "...{text1[start:end]}..."')
        print(f'  Doc2: "...{text2[start:end]}..."')
    
    # 判断为什么没被去重
    print("\n" + "="*80)
    print("WHY NOT DEDUPLICATED?")
    print("="*80)
    
    if not same_query:
        print("\n❌ REASON: Documents are in DIFFERENT queries")
        print("   Deduplication only works within the same query.")
        print("   Documents from different queries are never compared.")
    elif tfidf_similarity is not None and tfidf_similarity < 0.95:
        print(f"\n❌ REASON: TF-IDF similarity ({tfidf_similarity:.4f}) < threshold (0.95)")
        print(f"   The documents are {tfidf_similarity*100:.2f}% similar, but the threshold is 95%.")
        
        # 分析为什么相似度不够高
        print("\n   Possible reasons:")
        
        # 长度差异
        len_diff = abs(len(text1) - len(text2))
        len_ratio = min(len(text1), len(text2)) / max(len(text1), len(text2))
        print(f"   - Length difference: {len_diff} chars (ratio: {len_ratio:.2f})")
        
        # 词汇差异
        only_in_1 = words1 - words2
        only_in_2 = words2 - words1
        print(f"   - Words only in Doc1: {len(only_in_1)} ({', '.join(list(only_in_1)[:5])}...)")
        print(f"   - Words only in Doc2: {len(only_in_2)} ({', '.join(list(only_in_2)[:5])}...)")
        
        # TF-IDF权重差异
        if len(only_in_1) > 0 or len(only_in_2) > 0:
            print(f"   - These unique words have TF-IDF weights that lower the overall similarity")
    else:
        print(f"\n✓ Similarity ({tfidf_similarity:.4f}) >= threshold (0.95)")
        print("   These documents SHOULD have been deduplicated!")
        print("   This might be a bug or they were added at different stages.")
    
    # 建议
    print("\n" + "="*80)
    print("RECOMMENDATIONS:")
    print("="*80)
    
    if same_query and tfidf_similarity is not None:
        if tfidf_similarity >= 0.90 and tfidf_similarity < 0.95:
            print(f"\n💡 These documents are highly similar ({tfidf_similarity*100:.2f}%)")
            print(f"   Consider lowering the threshold to 0.90 to catch this case.")
        elif tfidf_similarity >= 0.95:
            print(f"\n⚠️  These documents should have been deduplicated!")
            print(f"   Please check the deduplication logic.")
        else:
            print(f"\n✓ The current threshold (0.95) is appropriate.")
            print(f"   These documents are only {tfidf_similarity*100:.2f}% similar.")


# ============================================================
# 运行分析
# ============================================================

if __name__ == "__main__":
    # 读取结果
    result = pd.read_csv('result_interleaved_with_text.csv')
    
    # 转换为字符串
    result['docno'] = result['docno'].astype(str)
    result['qid'] = result['qid'].astype(str)
    
    # 分析你提到的两个文档
    analyze_document_pair('6658615', '6351571', result)
    
    # 可选：检查这两个文档在原始df中的情况
    print("\n" + "="*80)
    print("CHECKING IN ORIGINAL DEDUPLICATED DFs:")
    print("="*80)
    
    # 读取去重后的df
    df_colbert = pd.read_csv('df_colbert_deduped.csv')
    df_colbert_prf = pd.read_csv('df_colbert_prf_deduped.csv')
    
    df_colbert['docno'] = df_colbert['docno'].astype(str)
    df_colbert_prf['docno'] = df_colbert_prf['docno'].astype(str)
    
    print("\nIn df_colbert (E2E):")
    doc_in_e2e = df_colbert[df_colbert['docno'].isin(['6658615', '6351571'])]
    if len(doc_in_e2e) > 0:
        print(doc_in_e2e[['qid', 'docno', 'score', 'rank']].to_string())
    else:
        print("  Not found")
    
    print("\nIn df_colbert_prf:")
    doc_in_prf = df_colbert_prf[df_colbert_prf['docno'].isin(['6658615', '6351571'])]
    if len(doc_in_prf) > 0:
        print(doc_in_prf[['qid', 'docno', 'score', 'rank']].to_string())
    else:
        print("  Not found")

ANALYZING DOCUMENTS: 6658615 vs 6351571

--------------------------------------------------------------------------------
DOCUMENT 1:
--------------------------------------------------------------------------------
DocNo: 6658615
QID: 104861
Rank: 4
Label: B-Only
Text length: 722 characters

Full text:
Because the cost of a cubic yard of concrete has topped $100.00 dollars per yard in a lot of areas around the U.S., it's helpful to know the exact cost in your area. The slab you see us pouring below used 120 yards of concrete.The cost per cubic yard of 3000 psi concrete for this slab was $92.00 dollars. That's $11,040.00 just for the concrete! The cost for a yard of concrete will vary in different parts of the country and all over the world.ay's Concrete Floors, Inc is the name of my company. We pour a lot of 3000 psi. concrete for interior concrete floors and 4000 psi concrete for exterior concrete. The concrete cost per yard of concrete is $92.00 for 3000 3/4 psi concrete and $100.00 